# L3b: Stacks and Queues

An array lets you read or write any element at any time. A stack and a queue give up that freedom: items enter and leave only at the ends, and the end you may touch decides the order work comes back. Today we build both on ordinary Julia arrays, wrap them in interfaces that enforce the rules, and meet the linked list, which stores order in references rather than positions and leads to the trees ahead.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Pick a stack or a queue:__ Say which item comes out first from a stack and which comes out first from a queue, and choose between them based on the order a problem needs. Say how an array keeps order by position and a linked list keeps order by having each item point to the next.
> * __Use the functions the type gives you:__ Add and remove items by calling those functions instead of touching the array inside the type. Explain why that keeps the order correct, and how it matches the function contracts from Week 2.
> * __Find the stack and the queue in code you run:__ Identify the call stack that tracks the calls a program has started but not yet finished, and the queue that a search uses to hold places it has not visited. Work out the order an algorithm will run in from which of the two it uses.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

This lecture needs nothing beyond the course environment: the [Test standard library](https://docs.julialang.org/en/v1/stdlib/Test/) supplies the checks we run along the way, the course package supplies the `MyStack` and `MyQueue` types and [the `isbalanced(...)` function](../../../code/src/StacksQueues.jl) we examine, and the `MutableLinkedList` type in the linked-list section comes from [the DataStructures.jl package](https://github.com/JuliaCollections/DataStructures.jl), which the course environment already carries.
___

## Stacks: last in, first out

Some work is naturally served most-recent-first. The undo feature of an editor reverses the latest edit, not the oldest one, and the back button of a browser returns to the page you just left. The structure behind every one of these is a stack.

> __The rule a stack follows:__
>
> A stack accepts new items and releases items at the same end, called the top. The last item pushed is the first item popped, so a stack replays history in reverse. This order is called __last-in-first-out__, or __LIFO__. Think of a stack of plates: you can only add or remove the top plate, and the last plate you put on is the first one you take off.

The schematic shows the rule on the values we will reuse all lecture: `push!(s, 16)` places 16 on top of the stack holding 8, 4, 2, and the very next `pop!(s)` removes and returns that same 16, the newest item.

<div>
    <center>
        <img src="figs/Fig-Stack.svg" width="560" alt="A stack holding 8, 4, 2 shown before and after push!(s, 16) places 16 on top, and after pop!(s) removes and returns that same 16"/>
    </center>
</div>

A plain Julia vector can already do both: [the `push!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.push!) adds an item at the back, and [the `pop!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.pop!) removes and returns the item at the back. Treat the back of the vector as the top of the stack, and undo falls out by itself. Let's record three edits and then undo two of them:

In [ ]:
let
    edit_history = Vector{String}() # the array behind an undo feature
    push!(edit_history, "typed the heading")
    push!(edit_history, "added the figure")
    push!(edit_history, "fixed the caption")
    (first_undo = pop!(edit_history), second_undo = pop!(edit_history), still_applied = edit_history)
end

The two undo steps came back in reverse order of entry: the caption fix first, then the figure, with the heading still applied. That is LIFO doing exactly what an undo feature needs.

There is a catch, though. Nothing about a plain vector enforces the rule: any caller can remove an element from the front, and the LIFO guarantee is gone.

The course package closes that hole with a `MyStack` type, defined in [the `StacksQueues.jl` file](../../../code/src/StacksQueues.jl). It is a [composite type](https://docs.julialang.org/en/v1/manual/types/#Composite-Types) holding the vector in a private `items` field, with public functions for pushing, popping, and peeking as the only supported access path. Julia does not lock fields away, so this is a convention rather than a wall, and code that stays on the supported path cannot break the order.

Let's point the `MyStack` type at a job from Week 2, walking the characters of a string, using the molecular formula of glucose as the string. The `formula_stack::MyStack{Char}` variable holds each character of `C6H12O6` in arrival order:

In [ ]:
formula_stack = let
    formula_stack = MyStack{Char}()
    for character in "C6H12O6"
        push!(formula_stack, character)
    end
    formula_stack
end

Popping until the stack is empty must now return the characters in reverse arrival order. The `reversed_formula::String` variable collects them:

In [ ]:
reversed_formula = let
    characters = Vector{Char}()
    while !isempty(formula_stack)
        push!(characters, pop!(formula_stack))
    end
    String(characters)
end

The formula comes back as `6O21H6C`: the final `6` was pushed last, so it is popped first. Reversal is what LIFO means, and it is why a stack is the right place for anything that has to come apart in the opposite order it was built.

Reading the formula back intact takes the other rule.
___

## Queues: first in, first out
Other work must be served in arrival order. A shared printer takes jobs in the order they were submitted, and a simulation processes events in the order they occur. Serving either of these most-recent-first would be wrong in an obvious way: latecomers would jump the line.

> __The rule a queue follows:__
>
> A queue accepts new items at the back and releases items from the front. The first item in is the first item out, so a queue preserves arrival order. This order is called __first-in-first-out__, or __FIFO__.

The schematic runs the same values through this rule: pushing 16 adds it at the back of the line behind 8, while the next removal serves the 2 at the front, exactly as a checkout line would.

<div>
    <center>
        <img src="figs/Fig-Queue.svg" width="500" alt="A queue holding 2, 4, 8 shown before and after push!(q, 16) joins the back of the line, and after popfirst!(q) serves and returns the 2 from the front"/>
    </center>
</div>

On a plain vector, [the `popfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.popfirst!) removes and returns the first element. The `MyQueue` type in [the `StacksQueues.jl` file](../../../code/src/StacksQueues.jl) wraps that rule the same way `MyStack` wrapped LIFO: internal vector, public functions, no other supported access path.

It also answers the glucose question the stack section left open. The `formula_reading::String` variable pushes the same formula through a `MyQueue{Char}` and drains it from the front:

In [ ]:
formula_reading = let
    character_queue = MyQueue{Char}()
    for character in "C6H12O6"
        push!(character_queue, character)
    end
    characters = Vector{Char}()
    while !isempty(character_queue)
        push!(characters, popfirst!(character_queue))
    end
    String(characters)
end

The queue reads the formula back exactly as written, `C6H12O6`, where the stack returned `6O21H6C`. Same characters, opposite readings. The only difference between the two runs is which end each rule allows you to touch.

> __What the two ends cost on a vector:__
>
> A Julia vector records where its data begins, so [the `popfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.popfirst!) removes the front element without moving the rest, at a cost that does not grow with length. Adding at the front is what costs: [the `pushfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.pushfirst!) must make room, and when there is none it moves everything already stored. A queue never has to do that, since it pushes at the back and pops at the front.

Code that does need a cheap front insertion reaches for a purpose-built structure such as the `Deque` type from [the DataStructures.jl package](https://github.com/JuliaCollections/DataStructures.jl), which keeps room at both ends. We will meet this build-versus-buy trade again on Thursday, when our hand-built sorts race Julia's library sort.

___

## Application: checking balanced delimiters
Every Julia expression you have typed this semester obeyed a rule you never checked by hand: parentheses, brackets, and braces must close in last-opened-first-closed order. Read that phrase again, because last-opened-first-closed is the stack rule, and a stack is exactly how a parser verifies delimiters.

> __The algorithm:__
>
> Walk the text one character at a time, keeping a stack of the delimiters currently open. An opener is pushed. A closer must match the most recently opened delimiter, so pop the stack and compare. A mismatch, a closer with nothing open, or anything left open at the end of the text each mean the text is unbalanced.

The `isbalanced(...)` function in [the `StacksQueues.jl` file](../../../code/src/StacksQueues.jl) implements this walk, and it shows the public-interface, private-implementation split in its smallest form. The function callers use is `isbalanced(...)`, while the `_OPENER_FOR_CLOSER` table and [the `_isopener(...)` helper function](../../../code/src/StacksQueues.jl) are private details, marked by the leading-underscore convention Julia programmers use for internals a caller should not rely on.

One caveat: the checker reads raw characters, so a lone delimiter inside a quoted string counts like any other. A real parser strips string literals and comments before a check like this runs.

Let's test `isbalanced(...)` with [the `@testset` macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@testset), including the failure cases that prove the checker can say no:

In [ ]:
@testset "isbalanced enforces stack discipline" begin
    @test isbalanced("f(x[2]) + {a: (b)}") # closes in last-opened-first-closed order
    @test isbalanced("no delimiters at all") # nothing to check is balanced
    @test !isbalanced("f(x[2)]") # closes the paren while the bracket is still open
    @test !isbalanced(")(") # closes a paren that was never opened
    @test !isbalanced("open( forever") # leaves a paren open at the end
end;

All five verdicts agree with a reading by eye, and the two things doing the work are what this lecture is about: a stack to remember what is open, and an interface that keeps the caller out of the details.
___

## Using a stack and a queue together: a replay buffer
The two rules are most useful together. Picture a survey robot exploring a planetary surface: mission control transmits movement commands, which the robot must execute in the order sent, and a `back` command must undo the most recent move. Execution order is a queue's job, and undo is a stack's. One stream of commands, two rules, each covering what the other cannot.

> __The design:__
>
> Commands wait in a `MyQueue{Char}` and are executed first-in-first-out, exactly as transmitted. Every executed move is also pushed onto a `MyStack{Char}` of history. A `back` command pops that history and executes the inverse of whatever comes off, so undo always applies to the most recent move, no matter when the command sequence was written.

The `replay_trace::NamedTuple` variable runs a small mission: north, north, east, east, west, then two `back` commands and a pause. Watch what each `back` undoes:

In [ ]:
replay_trace = let
    command_tape = MyQueue{Char}()
    for command in "nneewbbp" # north, north, east, east, west, back, back, pause
        push!(command_tape, command)
    end

    displacement = Dict('n' => (0, 1), 's' => (0, -1), 'e' => (1, 0), 'w' => (-1, 0), 'p' => (0, 0))
    inverse_move = Dict('n' => 's', 's' => 'n', 'e' => 'w', 'w' => 'e')

    undo_history = MyStack{Char}()
    x, y = 0, 0
    mission_log = Vector{String}()
    while !isempty(command_tape)
        command = popfirst!(command_tape) # commands execute in transmission order
        action = command
        if command == 'b'
            action = isempty(undo_history) ? 'p' : inverse_move[pop!(undo_history)]
        elseif haskey(inverse_move, command)
            push!(undo_history, command) # only real moves are undoable
        end
        (dx, dy) = displacement[action]
        x += dx
        y += dy
        push!(mission_log, "command $(command) -> action $(action), position ($(x), $(y))")
    end
    (final_position = (x, y), mission_log = mission_log)
end

The log shows the first `back` undoing the westward step and the second undoing an eastward one: most recent first, and that is the stack. The commands themselves ran in exactly the order they were sent, and that is the queue.

This coupling, a queue for what to do and a stack for what was done, is a common shape for a system that must run commands in order and undo them in reverse.
___

## The call stack
You have been using a stack all semester without seeing it. The calls a program has started but not yet finished are tracked on a stack: entering a call pushes a __stack frame__ holding the local variables and the point to return to, and returning pops that frame.

Calls therefore unwind in reverse order, and the most recently entered function is always the first to finish. That structure is the __call stack__, and it is a stack in exactly the sense of this lecture.

This is the model the language lets you reason with, not a promise about machine code: the compiler may inline a small call and never build a frame for it.

To watch this happen, let's nest three functions and print a line on the way into and out of each one:

In [ ]:
let
    inner() = println("        inner  entered last, finished first")
    function middle()
        println("    middle entered second")
        inner()
        println("    middle finished second")
    end
    function outer()
        println("outer  entered first")
        middle()
        println("outer  finished last")
    end
    outer()
end

The entry lines print in call order and the finish lines print in reverse, the same push-pop pattern as the undo demo. What the printout shows is that order, not a count of the frames actually built.

The frame budget is finite, and a chain of calls that never returns keeps pushing frames until the runtime gives up with a [`StackOverflowError` exception](https://docs.julialang.org/en/v1/base/base/#Core.StackOverflowError), an error named for this exact stack.
___

## Linked lists: order without an array

Our `MyStack` and `MyQueue` types kept their items in one contiguous array, and the queue's cost note showed the drawback: making room at the front of a vector means moving the elements already there. A __linked list__ is cheap exactly where an array is expensive.

Each item lives in its own small __node__, and a node holds a value and a reference to the node that follows it. The list itself remembers only the __head__, the first node. The last node references nothing, which is how a traversal knows to stop.

> __How each layout stores order:__
>
> An array stores order in positions: the fifth element sits five slots from the start, so jumping straight to it is instant, but making room at the front means moving everything. A linked list stores order in references: reaching the fifth element means following four links, but splicing a node in or out at a place you already hold is a couple of relinks, with nothing moved. Neither layout wins outright: each is fast where the other is slow.

<div>
    <center>
        <img src="figs/Fig-LinkedList.svg" width="700" alt="A linked list holding 2, 4, 8: the head references the first node, each node holds a value and a reference to the node that follows it, the last node references nothing, and a second row shows 16 spliced in after 4 with two relinks while nothing else moves"/>
    </center>
</div>


The schematic's splice is the operation arrays are slowest at, and a linked list does it by updating two references. It completes the story of the earlier figures: the stack pushed 16 on top, the queue added it at the back, and the linked list can put it anywhere.

We do not need to build the structure by hand. The `MutableLinkedList` type from [the DataStructures.jl package](https://github.com/JuliaCollections/DataStructures.jl) is a ready-made linked list, and it is __doubly linked__: each node also references the node before it, so a splice updates links in both directions rather than the one the schematic draws. The extra reference uses more memory and lets you walk the list backwards.

Let's load the glucose formula into one. The `formula_list::MutableLinkedList{Char}` variable holds one node per character, each linked after the last:

In [ ]:
formula_list = let
    formula_list = MutableLinkedList{Char}()
    for character in "C6H12O6"
        push!(formula_list, character) # each new node is linked after the current last node
    end
    formula_list
end

Iterating the list follows the references from the head, so the characters come back in arrival order, no array required. Adding at the front is now cheap, and that is the operation a vector is slow at.

The `front_edit::NamedTuple` variable pushes a stray character on with [the `pushfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.pushfirst!) and removes it with [the `popfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.popfirst!). Each operation is a constant amount of pointer work, whatever the length of the list, and then we read the whole list back:

In [ ]:
front_edit = let
    pushfirst!(formula_list, '?') # add at the front: constant pointer work, nothing shifts
    removed = popfirst!(formula_list) # remove it again; the existing nodes never move
    (removed = removed, reading = String(collect(formula_list)))
end

The reading is `C6H12O6`, intact and in order, and neither front operation moved another node.

This idea is where today's material grows next. A linked-list node references exactly one successor. Let nodes reference several, grown from a single root with no cycles, and the chain becomes a __tree__. Tomorrow's lecture draws a recursive computation as a tree of calls, and the active path through that tree lives on the call stack we just watched.

Next week, graph search walks structures built from this node-and-reference idea, holding its frontier in a container of your choosing. Make it a queue and the search spreads outward level by level, make it a stack and it dives deep before backing up. Same algorithm, different rule, different traversal.
___

## Summary
A stack and a queue are built on the same kind of storage under two different rules about which end you may touch, and the rule you pick decides the order work comes back.

> __Key Takeaways:__
>
> * __A stack and a queue are the same storage with different rules:__ Adding and removing at one end gives last-in-first-out, and adding at the back while removing at the front gives first-in-first-out. A linked list is storage rather than a third rule, so either rule can be built on one, trading a walk to reach a position for cheap splicing.
> * __The type's functions are what keep the order correct:__ Every operation on our stack and queue goes through a function that respects the order, and the vector inside is left alone by convention. Code that only calls those functions cannot break the order.
> * __Where stacks and queues show up:__ A language runtime tracks unfinished calls on a stack, a delimiter checker tracks what is still open on a stack, and a stream of commands that must run in the order sent is a queue. Knowing which of the two an algorithm uses tells you the order it will do its work in.

Pick the rule from the order the problem requires, and pick the storage from the operations it will perform most often.
___